# Phase 2 — Exploring the FPL Public API

Goal: understand the data **before** we write any production code.

The official Fantasy Premier League website exposes a free JSON API at:

```
https://fantasy.premierleague.com/api/
```

We'll hit four endpoints:

| Endpoint | What it returns |
|---|---|
| `bootstrap-static/` | All players, teams, gameweeks, positions |
| `fixtures/` | Every fixture in the season with difficulty ratings |
| `element-summary/{id}/` | One player's full GW-by-GW history |
| `entry/{team_id}/event/{gw}/picks/` | A manager's squad for a given GW |

By the end of this notebook you should be able to **explain out loud** what each endpoint returns and where every feature we'll use later comes from.

In [2]:
import requests
import pandas as pd

BASE = "https://fantasy.premierleague.com/api"
HEADERS = {"User-Agent": "fpl-ai-exploration/0.1"}

def get(path: str):
    """Tiny helper that hits the FPL API and returns parsed JSON."""
    url = f"{BASE}/{path.lstrip('/')}"
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return r.json()

## 1. The `bootstrap-static` endpoint — the master file

This single call gives us almost everything *static* about the season.

In [3]:
boot = get("bootstrap-static/")
print("Top-level keys:")
for k in boot.keys():
    v = boot[k]
    size = len(v) if hasattr(v, "__len__") else "-"
    print(f"  {k:25s}  type={type(v).__name__:10s}  len={size}")

Top-level keys:
  chips                      type=list        len=8
  events                     type=list        len=38
  game_settings              type=dict        len=35
  game_config                type=dict        len=3
  phases                     type=list        len=11
  teams                      type=list        len=20
  total_players              type=int         len=-
  element_stats              type=list        len=26
  element_types              type=list        len=4
  elements                   type=list        len=832


**Q1 — How many players are there?**

In [4]:
print(f"Number of players: {len(boot['elements'])}")
print(f"Number of teams:   {len(boot['teams'])}")
print(f"Number of GWs:     {len(boot['events'])}")
print(f"Position types:    {len(boot['element_types'])}")

Number of players: 832
Number of teams:   20
Number of GWs:     38
Position types:    4


**Q2 — What columns describe a player?**

In [5]:
player = boot['elements'][0]
print(f"Total fields per player: {len(player)}\n")
print("Sample player record:")
for k, v in list(player.items())[:25]:
    print(f"  {k:35s}  {v}")
print("  ... (more fields below)")

Total fields per player: 105

Sample player record:
  can_transact                         True
  can_select                           True
  chance_of_playing_next_round         None
  chance_of_playing_this_round         None
  code                                 154561
  cost_change_event                    1
  cost_change_event_fall               -1
  cost_change_start                    6
  cost_change_start_fall               -6
  price_change_percent                 0
  dreamteam_count                      2
  element_type                         1
  ep_next                              5.5
  ep_this                              5.5
  event_points                         6
  first_name                           David
  form                                 4.5
  id                                   1
  in_dreamteam                         True
  news                                 
  news_added                           None
  now_cost                             61
  photo    

In [6]:
# Easier to scan as a DataFrame
players_df = pd.DataFrame(boot['elements'])
print("All player columns:")
print(list(players_df.columns))

All player columns:
['can_transact', 'can_select', 'chance_of_playing_next_round', 'chance_of_playing_this_round', 'code', 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'price_change_percent', 'dreamteam_count', 'element_type', 'ep_next', 'ep_this', 'event_points', 'first_name', 'form', 'id', 'in_dreamteam', 'news', 'news_added', 'now_cost', 'photo', 'points_per_game', 'removed', 'second_name', 'selected_by_percent', 'special', 'squad_number', 'status', 'team', 'team_code', 'total_points', 'transfers_in', 'transfers_in_event', 'transfers_out', 'transfers_out_event', 'value_form', 'value_season', 'web_name', 'known_name', 'region', 'team_join_date', 'birth_date', 'has_temporary_code', 'opta_code', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index', 'clearances_blocks

**Key columns we'll use later:**

| Column | Meaning |
|---|---|
| `id` | Unique player id (use this for `element-summary/{id}/`) |
| `web_name` | Short display name (e.g. "Salah") |
| `team` | The player's club (id, joins to `boot['teams']`) |
| `element_type` | Position id (joins to `boot['element_types']`) |
| `now_cost` | **Price × 10** (75 means £7.5m) |
| `total_points` | Season points so far |
| `form` | Average points over last 30 days |
| `minutes`, `goals_scored`, `assists`, `clean_sheets`, `bonus`, `bps` | Raw stats |
| `ict_index`, `influence`, `creativity`, `threat` | FPL composite scores |
| `expected_goals`, `expected_assists`, `expected_goal_involvements` | Underlying xG/xA |
| `status` | `a`=available, `i`=injured, `d`=doubt, `s`=suspended |
| `chance_of_playing_next_round` | 0–100 |

**Q3 — What does `element_type` mean?**

In [7]:
positions_df = pd.DataFrame(boot['element_types'])
positions_df[['id', 'singular_name', 'singular_name_short', 'squad_select', 'squad_min_play', 'squad_max_play']]

,id,singular_name,singular_name_short,squad_select,squad_min_play,squad_max_play
0,1,Goalkeeper,GKP,2,1,1
1,2,Defender,DEF,5,3,5
2,3,Midfielder,MID,5,2,5
3,4,Forward,FWD,3,1,3


So `element_type`:
- `1` = Goalkeeper (GKP)
- `2` = Defender (DEF)
- `3` = Midfielder (MID)
- `4` = Forward (FWD)

Notice `squad_select` confirms FPL squad rules: **2 GK, 5 DEF, 5 MID, 3 FWD = 15 players** — we'll use this in Phase 6.

**Q4 — What does `now_cost` mean?**

In [8]:
expensive = players_df.nlargest(5, 'now_cost')[['web_name', 'now_cost']].copy()
expensive['price_£m'] = expensive['now_cost'] / 10
expensive

,web_name,now_cost,price_£m
519,Haaland,146,14.6
465,M.Salah,140,14.0
280,Palmer,104,10.4
546,B.Fernandes,104,10.4
481,Isak,103,10.3


Prices are stored as **integers × 10** to avoid floating-point issues. We'll always divide by 10 when displaying.

**Q5 — What's in `events`? Find the current gameweek.**

In [9]:
events_df = pd.DataFrame(boot['events'])
print("Event columns:", list(events_df.columns)[:15], "...\n")
events_df[['id', 'name', 'deadline_time', 'finished', 'is_current', 'is_next', 'average_entry_score']].head(10)

Event columns: ['id', 'name', 'deadline_time', 'release_time', 'average_entry_score', 'finished', 'data_checked', 'highest_scoring_entry', 'deadline_time_epoch', 'deadline_time_game_offset', 'highest_score', 'is_previous', 'is_current', 'is_next', 'cup_leagues_created'] ...



,id,name,deadline_time,finished,is_current,is_next,average_entry_score
0,1,Gameweek 1,2025-08-15T17:30:00Z,True,False,False,54
1,2,Gameweek 2,2025-08-22T17:30:00Z,True,False,False,51
2,3,Gameweek 3,2025-08-30T10:00:00Z,True,False,False,48
3,4,Gameweek 4,2025-09-13T10:00:00Z,True,False,False,63
4,5,Gameweek 5,2025-09-20T10:00:00Z,True,False,False,42
5,6,Gameweek 6,2025-09-27T10:00:00Z,True,False,False,46
6,7,Gameweek 7,2025-10-03T17:30:00Z,True,False,False,60
7,8,Gameweek 8,2025-10-18T10:00:00Z,True,False,False,56
8,9,Gameweek 9,2025-10-24T17:30:00Z,True,False,False,46
9,10,Gameweek 10,2025-11-01T13:30:00Z,True,False,False,65


In [10]:
# Find the current and next GW
current = events_df[events_df['is_current']].iloc[0] if events_df['is_current'].any() else None
next_gw = events_df[events_df['is_next']].iloc[0] if events_df['is_next'].any() else None
print(f"Current GW: {current['id'] if current is not None else 'none (between seasons)'}")
print(f"Next GW:    {next_gw['id'] if next_gw is not None else 'none'}")
print("\nThis is the helper logic we'll put into FPLClient in Phase 3.")

Current GW: 35
Next GW:    36

This is the helper logic we'll put into FPLClient in Phase 3.


**Q6 — What's in `teams`?**

In [11]:
teams_df = pd.DataFrame(boot['teams'])
teams_df[['id', 'name', 'short_name', 'strength',
          'strength_overall_home', 'strength_overall_away',
          'strength_attack_home', 'strength_attack_away',
          'strength_defence_home', 'strength_defence_away']].head(10)

,id,name,short_name,strength,strength_overall_home,strength_overall_away,strength_attack_home,strength_attack_away,strength_defence_home,strength_defence_away
0,1,Arsenal,ARS,5,1305,1355,1340,1390,1270,1320
1,2,Aston Villa,AVL,3,1135,1230,1120,1210,1150,1250
2,3,Burnley,BUR,2,975,1045,910,1050,1040,1040
3,4,Bournemouth,BOU,3,1150,1235,1060,1230,1240,1240
4,5,Brentford,BRE,3,1135,1215,1120,1160,1150,1270
5,6,Brighton,BHA,4,1170,1250,1140,1220,1200,1280
6,7,Chelsea,CHE,3,1190,1195,1130,1140,1250,1250
7,8,Crystal Palace,CRY,3,1160,1185,1150,1200,1170,1170
8,9,Everton,EVE,3,1135,1160,1130,1150,1140,1170
9,10,Fulham,FUL,3,1105,1200,1090,1170,1120,1230


These **strength ratings (1–5)** are extremely useful features for the model — they encode "how hard is it to score against this team". We'll join them onto fixtures in Phase 4.

## 2. The per-player history endpoint — the goldmine

`bootstrap-static` only gives season totals. To train a model we need **per-gameweek** history.

In [12]:
# Pick a high-profile player so the history table is interesting
salah = players_df[players_df['web_name'].str.contains('Salah', case=False, na=False)].iloc[0]
pid = int(salah['id'])
print(f"Looking at player_id={pid}, name={salah['web_name']}")

summary = get(f"element-summary/{pid}/")
print("\nKeys in element-summary:", list(summary.keys()))
print(f"  history       — {len(summary['history'])} rows (this season, one per GW played)")
print(f"  fixtures      — {len(summary['fixtures'])} rows (upcoming fixtures)")
print(f"  history_past  — {len(summary['history_past'])} rows (career season totals)")

Looking at player_id=381, name=M.Salah

Keys in element-summary: ['fixtures', 'history', 'history_past']
  history       — 35 rows (this season, one per GW played)
  fixtures      — 3 rows (upcoming fixtures)
  history_past  — 10 rows (career season totals)


In [13]:
history_df = pd.DataFrame(summary['history'])
print("history columns:", list(history_df.columns))
history_df[['round', 'opponent_team', 'was_home', 'minutes', 'total_points',
            'goals_scored', 'assists', 'clean_sheets', 'bonus', 'bps',
            'ict_index', 'expected_goals', 'expected_assists']].head(10)

history columns: ['element', 'fixture', 'opponent_team', 'total_points', 'was_home', 'kickoff_time', 'team_h_score', 'team_a_score', 'round', 'modified', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index', 'clearances_blocks_interceptions', 'recoveries', 'tackles', 'defensive_contribution', 'starts', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded', 'value', 'transfers_balance', 'selected', 'transfers_in', 'transfers_out']


,round,opponent_team,was_home,minutes,total_points,goals_scored,assists,clean_sheets,bonus,bps,ict_index,expected_goals,expected_assists
0,1,4,True,90,8,1,0,0,1,36,13.4,0.26,0.11
1,2,15,False,90,5,0,1,0,0,24,5.5,0.00,0.21
2,3,1,True,90,3,0,0,1,0,5,2.4,0.05,0.03
3,4,3,False,90,9,1,0,1,1,26,8.3,0.79,0.15
4,5,9,True,90,5,0,1,0,0,14,6.5,0.12,0.19
5,6,8,False,90,2,0,0,0,0,-2,6.4,0.49,0.04
6,7,7,False,90,2,0,0,0,0,4,6.7,0.26,0.27
7,8,14,True,84,2,0,0,0,0,10,10.2,0.41,0.14
8,9,5,False,90,6,1,0,0,0,15,10.7,0.68,0.06
9,10,2,True,90,10,1,0,1,2,31,8.8,0.22,0.13


**This is the table our ML model will train on.**

- `round` = gameweek number (1–38) → we use this for our **time-based train/validation split**.
- `total_points` = the **label** (what we're predicting).
- Everything else = potential **features**, but we'll only use **lagged** versions (e.g. average over the previous 3 GWs) to avoid data leakage in Phase 4.

## 3. The fixtures endpoint — opponent difficulty

In [14]:
fixtures = get("fixtures/")
fixtures_df = pd.DataFrame(fixtures)
print("Total fixtures:", len(fixtures_df))
print("Columns:", list(fixtures_df.columns))
fixtures_df[['event', 'team_h', 'team_a',
             'team_h_difficulty', 'team_a_difficulty',
             'finished', 'kickoff_time']].head(10)

Total fixtures: 380
Columns: ['code', 'event', 'finished', 'finished_provisional', 'id', 'kickoff_time', 'minutes', 'provisional_start_time', 'started', 'team_a', 'team_a_score', 'team_h', 'team_h_score', 'stats', 'team_h_difficulty', 'team_a_difficulty', 'pulse_id']


,event,team_h,team_a,team_h_difficulty,team_a_difficulty,finished,kickoff_time
0,1,12,4,3,4,True,2025-08-15T19:00:00Z
1,1,2,15,3,4,True,2025-08-16T11:30:00Z
2,1,6,10,3,4,True,2025-08-16T14:00:00Z
3,1,18,3,1,3,True,2025-08-16T14:00:00Z
4,1,17,19,2,3,True,2025-08-16T14:00:00Z
5,1,20,13,4,2,True,2025-08-16T16:30:00Z
6,1,7,8,3,3,True,2025-08-17T13:00:00Z
7,1,16,5,3,3,True,2025-08-17T13:00:00Z
8,1,14,1,4,4,True,2025-08-17T15:30:00Z
9,1,11,9,3,3,True,2025-08-18T19:00:00Z


Fixture difficulty:
- `team_h_difficulty` = how hard the **home team** finds this match (1=easy, 5=very hard).
- `team_a_difficulty` = same from the **away team's** perspective.
- `event` = gameweek number; `finished=False` means it hasn't been played yet.

In Phase 4 we'll attach the upcoming fixture difficulty to each player's row before predicting.

## 4. The manager picks endpoint — your team

Optional, used in Phase 6 for **transfer recommendations**.

Find your team id in the FPL website URL — go to *Points*, the URL looks like
`https://fantasy.premierleague.com/entry/<TEAM_ID>/event/30`.

In [15]:
# Replace with your own team id once you know it. Public sample left blank.
TEAM_ID = 271610  # e.g. 12345

if TEAM_ID is not None and current is not None:
    picks = get(f"entry/{TEAM_ID}/event/{int(current['id'])}/picks/")
    picks_df = pd.DataFrame(picks['picks'])
    # Drop 'element_type' from the right side - the picks endpoint already
    # includes it, so a merge would create element_type_x / element_type_y.
    picks_df = picks_df.merge(
        players_df[['id', 'web_name', 'now_cost']],
        left_on='element', right_on='id', how='left'
    )
    picks_df['price_£m'] = picks_df['now_cost'] / 10
    display(picks_df[['position', 'web_name', 'element_type',
                      'price_£m', 'multiplier', 'is_captain', 'is_vice_captain']])
else:
    print("Set TEAM_ID above to see your squad.")

,position,web_name,element_type,price_£m,multiplier,is_captain,is_vice_captain
0,1,Donnarumma,1,5.6,1,False,False
1,2,Gabriel,2,7.2,1,False,False
2,3,Virgil,2,6.2,1,False,False
3,4,Hill,2,4.2,1,False,False
4,5,Gibbs-White,3,7.7,1,False,False
5,6,B.Fernandes,3,10.4,1,False,False
6,7,Anderson,3,5.6,1,False,False
7,8,Groß,3,5.5,1,False,False
8,9,Thiago,4,7.4,1,False,False
9,10,Bowen,4,7.8,2,True,False


## 5. Putting it together — what we know now

We can now build a player-level table that joins everything we'll need:

In [16]:
# Tiny preview of the shape we'll produce in Phase 4
preview = (
    players_df[['id', 'web_name', 'team', 'element_type', 'now_cost',
                'total_points', 'form', 'ict_index', 'status']]
    .merge(teams_df[['id', 'short_name']], left_on='team', right_on='id', suffixes=('', '_team'))
    .merge(positions_df[['id', 'singular_name_short']], left_on='element_type', right_on='id', suffixes=('', '_pos'))
    .rename(columns={'short_name': 'team_name', 'singular_name_short': 'position'})
    .drop(columns=['id_team', 'id_pos'])
)
preview['price_£m'] = preview['now_cost'] / 10
preview.nlargest(15, 'total_points')[['web_name', 'team_name', 'position', 'price_£m', 'total_points', 'form']]

,web_name,team_name,position,price_£m,total_points,form
519,Haaland,MCI,FWD,14.6,219,5.5
546,B.Fernandes,MUN,MID,10.4,209,5.0
4,Gabriel,ARS,DEF,7.2,191,4.5
489,Semenyo,MCI,MID,8.1,183,2.2
19,Rice,ARS,MID,7.2,176,3.2
209,Thiago,BRE,FWD,7.4,175,5.5
292,João Pedro,CHE,FWD,7.5,174,2.5
626,Gibbs-White,NFO,MID,7.7,172,9.8
770,Bowen,WHU,FWD,7.8,171,7.0
552,Casemiro,MUN,MID,5.8,164,7.5


## ✅ Phase 2 takeaways

1. **`bootstrap-static/`** = master snapshot (~600 players, 20 teams, 38 GWs, positions). One call gets nearly everything *static*.
2. **`element-summary/{id}/`** = per-player per-GW history (one call per player → ~600 calls; cache the result).
3. **`fixtures/`** = all matches with difficulty ratings.
4. **`entry/{id}/event/{gw}/picks/`** = a manager's current squad.
5. Prices are integers (`now_cost / 10` for £m).
6. Position ids: 1=GK, 2=DEF, 3=MID, 4=FWD; squad shape locked at 2/5/5/3.
7. Team strength ratings (1–5) split by home/away and attack/defence — strong features for the model.
8. Use `round` (gameweek number) for **time-based** train/validation splits — never random.

We're now ready for **Phase 3**: wrap these endpoints into a clean reusable `FPLClient` class in `src/fpl_api.py`.